# 02 — Limpeza e padronização dos dados

Transforma a camada **bronze** (`data/parquet/`, saída da ingestão) na camada **silver** (`data/silver/`), que é a
entrada do record linkage.

Princípios:
- **Mesmas funções nos dois lados** (busca e CNEFE): só assim as chaves de comparação são compatíveis.
- **Colunas originais preservadas**; tudo que é derivado ganha sufixo `_norm` ou nome próprio.
- **Notebook autocontido** (não importa nada de `src/`) e **idempotente** (`overwrite`): pode ser reexecutado do
  zero com `make clean-data`.

Decisões vindas da EDA (`01_analise_exploratoria.ipynb`):

| Achado | Tratamento |
|---|---|
| `COD_SETOR` sempre tem 16 caracteres terminando em `P` | `setor` = 15 primeiros dígitos |
| 9,05 M linhas para 8,53 M `COD_UNICO_ENDERECO` | deduplicar por `COD_UNICO_ENDERECO` |
| CNEFE mistura `2 DE JULHO` e `DOIS DE JULHO` | números por extenso → dígitos, nos dois lados |
| Título às vezes em `NOM_TITULO_SEGLOGR` (`SENHOR` + `DO BONFIM`), às vezes no nome | `nome_norm` = título + nome |
| Busca: prefixos-lixo (`None`, `10R`), tipo duplicado (`RUA FAZ ...`) | remover prefixo; guardar `tipo_secundario` + `nome_alternativo` |
| `SN`, `S/N`, vazio, `0` no número | `numero` nulo + `sem_numero` |
| 89/107 CEPs genéricos (`xxxxx000`) | `cep_generico` |
| `BOMFIM` × `BONFIM`, `CASSIMIRO` × `CASIMIRO` | chave fonética simplificada (`nome_fonetico`) |

In [1]:
from pathlib import Path

import pandas as pd
from pyspark.sql import Column, SparkSession
from pyspark.sql import functions as F

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRONZE = ROOT / "data" / "parquet"
SILVER = ROOT / "data" / "silver"
for path in (BRONZE / "base_busca", BRONZE / "cnefe_bahia"):
    assert (path / "_SUCCESS").exists(), f"{path} não encontrado: execute `make ingest` antes."

spark = (
    SparkSession.builder.appName("02-limpeza-dados")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "48")
    .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

busca_bronze = spark.read.parquet(str(BRONZE / "base_busca"))
cnefe_bronze = spark.read.parquet(str(BRONZE / "cnefe_bahia"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/11 01:54:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Dicionários de padronização

Mantidos curtos e explícitos. Os **tipos de logradouro válidos** são derivados do próprio CNEFE (seção 2.3), não
digitados à mão.

In [2]:
# Abreviações de tipo — aplicadas só ao 1º/2º token do logradouro (evita trocar um "R" no meio do nome).
ABREV_TIPO = {
    "R": "RUA", "AV": "AVENIDA", "AVN": "AVENIDA", "TV": "TRAVESSA", "TRAV": "TRAVESSA", "TR": "TRAVESSA",
    "ROD": "RODOVIA", "EST": "ESTRADA", "ESTR": "ESTRADA", "FAZ": "FAZENDA", "FZ": "FAZENDA",
    "POV": "POVOADO", "PC": "PRACA", "PCA": "PRACA", "PRC": "PRACA", "AL": "ALAMEDA", "LOT": "LOTEAMENTO",
    "CJ": "CONJUNTO", "CONJ": "CONJUNTO", "COND": "CONDOMINIO", "LD": "LADEIRA", "LAD": "LADEIRA",
    "VL": "VILA", "SIT": "SITIO", "DIST": "DISTRITO", "LG": "LARGO", "BC": "BECO", "CAM": "CAMINHO",
    "QD": "QUADRA", "COM": "COMUNIDADE", "ASSENT": "ASSENTAMENTO",
}

# Tipos aceitos como "tipo embutido" no 2º token (RUA FAZ RIO DA VARA). Lista curta para não confundir com
# nomes que também são tipos no CNEFE (RUA RIO BAIANO, RUA ALTO ...).
TIPOS_SECUNDARIOS = [
    "FAZENDA", "SITIO", "CAMINHO", "TRAVESSA", "AVENIDA", "POVOADO", "DISTRITO", "CONJUNTO",
    "LOTEAMENTO", "CONDOMINIO", "QUADRA", "ESTRADA", "RODOVIA", "CHACARA",
]

# Abreviações de títulos/palavras — aplicadas a qualquer token do nome.
ABREV_PALAVRA = {
    "DR": "DOUTOR", "DRA": "DOUTORA", "ENG": "ENGENHEIRO", "PROF": "PROFESSOR", "PROFA": "PROFESSORA",
    "PE": "PADRE", "CEL": "CORONEL", "STO": "SANTO", "STA": "SANTA", "SR": "SENHOR", "SRA": "SENHORA",
    "PRES": "PRESIDENTE", "GOV": "GOVERNADOR", "DEP": "DEPUTADO", "VER": "VEREADOR", "GAL": "GENERAL",
    "GEN": "GENERAL", "MAL": "MARECHAL", "ALM": "ALMIRANTE", "DES": "DESEMBARGADOR", "TEN": "TENENTE",
    "CAP": "CAPITAO", "MONS": "MONSENHOR", "JD": "JARDIM", "RESID": "RESIDENCIAL", "HAB": "HABITACIONAL",
}

# Números por extenso → dígitos (o CNEFE mistura "2 DE JULHO" e "DOIS DE JULHO").
NUMEROS_EXTENSO = {
    "PRIMEIRO": "1", "UM": "1", "DOIS": "2", "TRES": "3", "QUATRO": "4", "CINCO": "5", "SEIS": "6",
    "SETE": "7", "OITO": "8", "NOVE": "9", "DEZ": "10", "ONZE": "11", "DOZE": "12", "TREZE": "13",
    "QUATORZE": "14", "CATORZE": "14", "QUINZE": "15", "DEZESSEIS": "16", "DEZESSETE": "17",
    "DEZOITO": "18", "DEZENOVE": "19", "VINTE": "20", "TRINTA": "30",
}

STOPWORDS = ["DE", "DA", "DO", "DAS", "DOS", "E", "D"]
TOKENS_LIXO = ["NONE", "NULL", "NAN"]
MARCADORES_ZONA_RURAL = ["ZONA RURAL", "RURAL", "AREA RURAL"]
COMPLEMENTOS_SEM_VALOR = ["SN", "S N", "SEM NUMERO", "CASA", "E", "T", "A", "PROX"]

## 2. Funções de normalização (expressões Spark, sem UDF)

In [3]:
ACENTOS_DE, ACENTOS_PARA = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ", "AAAAAEEEEIIIIOOOOOUUUUCN"


def _mapa(dicionario: dict[str, str]) -> Column:
    return F.create_map([F.lit(x) for par in dicionario.items() for x in par])


def _colapsar(c: Column) -> Column:
    c = F.trim(F.regexp_replace(c, r"\s+", " "))
    return F.when(c == "", None).otherwise(c)


def normalizar_texto(c: Column) -> Column:
    """Maiúsculas, sem acento/pontuação, sem tokens-lixo, espaços colapsados; vazio → nulo."""
    c = F.translate(F.upper(c), ACENTOS_DE, ACENTOS_PARA)
    c = F.regexp_replace(c, r"'", "")                      # D'AJUDA → DAJUDA (forma usada no CNEFE)
    c = F.regexp_replace(c, r"[^A-Z0-9 ]", " ")
    c = F.regexp_replace(c, r"\b(" + "|".join(TOKENS_LIXO) + r")\b", " ")
    c = F.regexp_replace(c, r"\bD (?=[AEIOU])", "D")        # D AGUA → DAGUA
    return _colapsar(c)


def mapear_tokens(c: Column, dicionario: dict[str, str]) -> Column:
    m = _mapa(dicionario)
    tokens = F.transform(F.split(c, " "), lambda t: F.coalesce(F.element_at(m, t), t))
    return F.array_join(tokens, " ")


def normalizar_nome(c: Column) -> Column:
    """Texto já normalizado → expande abreviações de títulos e converte números por extenso."""
    c = mapear_tokens(c, {**ABREV_PALAVRA, **NUMEROS_EXTENSO})
    c = F.regexp_replace(c, r"\b([23])0 (?:E )?([1-9])\b", "$1$2")   # VINTE E UM → 20 E 1 → 21
    return _colapsar(c)


def chave_nome(c: Column) -> Column:
    """Nome sem stopwords: chave para comparação exata tolerante a 'DE/DA/DO'.

    Se o nome for só stopword (RUA E, RUA D), mantém o nome original.
    """
    return F.coalesce(_colapsar(F.regexp_replace(c, r"\b(" + "|".join(STOPWORDS) + r")\b", " ")), c)


REGRAS_FONETICAS = [
    (r"CH|SH", "X"), (r"PH", "F"), (r"LH", "L"), (r"NH", "N"),
    (r"QU(?=[EI])", "K"), (r"Q", "K"), (r"C(?=[EI])", "S"), (r"C", "K"),
    (r"G(?=[EI])", "J"), (r"Y", "I"), (r"W", "V"), (r"Z", "S"), (r"H", ""),
    (r"M(?=[^AEIOUBP ])", "N"),   # BOMFIM → BONFIM
    (r"(\w)\1+", "$1"),           # CASSIMIRO → CASIMIRO
]


def chave_fonetica(c: Column) -> Column:
    """Chave fonética simplificada para o português (aplicada sobre `chave_nome`)."""
    for padrao, troca in REGRAS_FONETICAS:
        c = F.regexp_replace(c, padrao, troca)
    return _colapsar(c)


def em(lista: list[str], c: Column) -> Column:
    return F.array_contains(F.array(*[F.lit(x) for x in lista]), c)

### 2.1 Teste rápido das funções com casos reais

In [4]:
casos = [
    "None EVILASIO FIRMATO", "RUA VINTE UM DE ABRIL", "DISTRITO 2 DE JULHO", "RUA SENHOR DO BOMFIM",
    "RUA CASSIMIRO DE ABREU", "CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES", "DISTRITO ENG PONTES",
    "RUA DA EXPANSAO CAIXA D'AGUA", "Avenida São João", "RUA E",
]
teste = spark.createDataFrame([(c,) for c in casos], "entrada string")
teste = teste.withColumn("texto", normalizar_texto(F.col("entrada"))) \
    .withColumn("nome", normalizar_nome(F.col("texto"))) \
    .withColumn("chave", chave_nome(F.col("nome"))) \
    .withColumn("fonetica", chave_fonetica(F.col("chave")))
teste.toPandas()

,entrada,texto,nome,chave,fonetica
0,None EVILASIO FIRMATO,EVILASIO FIRMATO,EVILASIO FIRMATO,EVILASIO FIRMATO,EVILASIO FIRMATO
1,RUA VINTE UM DE ABRIL,RUA VINTE UM DE ABRIL,RUA 21 DE ABRIL,RUA 21 ABRIL,RUA 21 ABRIL
2,DISTRITO 2 DE JULHO,DISTRITO 2 DE JULHO,DISTRITO 2 DE JULHO,DISTRITO 2 JULHO,DISTRITO 2 JULO
3,RUA SENHOR DO BOMFIM,RUA SENHOR DO BOMFIM,RUA SENHOR DO BOMFIM,RUA SENHOR BOMFIM,RUA SENOR BONFIM
4,RUA CASSIMIRO DE ABREU,RUA CASSIMIRO DE ABREU,RUA CASSIMIRO DE ABREU,RUA CASSIMIRO ABREU,RUA KASIMIRO ABREU
5,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,KONDOMINIO RESI DOUTOR ANTONIO KARLOS GOES
6,DISTRITO ENG PONTES,DISTRITO ENG PONTES,DISTRITO ENGENHEIRO PONTES,DISTRITO ENGENHEIRO PONTES,DISTRITO ENJENEIRO PONTES
7,RUA DA EXPANSAO CAIXA D'AGUA,RUA DA EXPANSAO CAIXA DAGUA,RUA DA EXPANSAO CAIXA DAGUA,RUA EXPANSAO CAIXA DAGUA,RUA EXPANSAO KAIXA DAGUA
8,Avenida São João,AVENIDA SAO JOAO,AVENIDA SAO JOAO,AVENIDA SAO JOAO,AVENIDA SAO JOAO
9,RUA E,RUA E,RUA E,RUA,RUA


### 2.2 Tipos de logradouro válidos (derivados do CNEFE)

In [5]:
# 1º token do tipo, ignorando ordinais ("1A TRAVESSA DA RUA" → TRAVESSA). Tipos com pelo menos 100 endereços.
TIPO_BASE_REGEX = r"^(?:\d+A )?(\S+)"
tipos_cnefe = (
    cnefe_bronze.select(F.regexp_extract(normalizar_texto(F.col("NOM_TIPO_SEGLOGR")), TIPO_BASE_REGEX, 1).alias("tipo"))
    .groupBy("tipo").count().where("count >= 100").orderBy(F.desc("count"))
    .toPandas()
)
TIPOS_VALIDOS = sorted(set(tipos_cnefe["tipo"]) | set(ABREV_TIPO.values()))
print(len(TIPOS_VALIDOS), "tipos válidos")
print(", ".join(TIPOS_VALIDOS))

164 tipos válidos
ACAMPAMENTO, ACESSO, ACUDE, AEROPORTO, AGLOMERADO, AGROVILA, ALAMEDA, ALDEIA, ALTO, ANEL, AREA, ARRAIAL, ASSENTAMENTO, ATALHO, AVENIDA, BAIA, BAIRRO, BAIXA, BAIXADA, BAIXAO, BARRA, BARRAGEM, BECO, BEIRA, BEIRADA, BLOCO, BOQUEIRAO, BREJO, CABECEIRA, CACHOEIRA, CALCADA, CAMINHO, CAMPO, CANAL, CAPAO, CENTRO, CERCA, CHACARA, CHAPADA, COHAB, COLONIA, COMUNIDADE, CONDOMINIO, CONJUNTO, CONTINUACAO, CONTORNO, CORREDOR, CORREGO, CRUZAMENTO, DESCIDA, DESVIO, DISTRITO, DIVISA, EDIFICIO, EIXO, ENGENHO, ENTRADA, ENTRONCAMENTO, ESCADA, ESCADARIA, ESTACAO, ESTANCIA, ESTRADA, EXTENSAO, FAVELA, FAZENDA, FINAL, FLORESTA, FUNDOS, GAMELEIRA, GARIMPO, GROTA, HABITACIONAL, HARAS, ILHA, INVASAO, JARDIM, LADEIRA, LADO, LAGO, LAGOA, LARGO, LATERAL, LIGACAO, LIMITE, LINHA, LOTE, LOTEAMENTO, LUGAREJO, MARGEM, MARGENS, MATA, MATO, MONTE, MORRO, NUCLEO, ORLA, OUTROS, PARALELA, PARQUE, PASSAGEM, PASSARELA, PASSEIO, PEDRA, PERIMETRO, PONTA, PONTE, PONTO, PORTO, POSTO, POVOADO, PRACA, PRAIA, PROJETA

## 3. Base de busca → silver

Passos do logradouro:
1. `normalizar_texto` (remove `None`, pontuação, acentos);
2. remove **prefixo-lixo**: 1º token não é tipo mas o 2º é (`10R FAZENDA ...`);
3. separa **tipo** (1º token, com abreviação expandida) e **nome**;
4. se o 1º token do nome também é um tipo (`RUA FAZ RIO DA VARA`), guarda `tipo_secundario` e `nome_alternativo`.

In [6]:
mapa_tipo = _mapa(ABREV_TIPO)


def tipo_de(token: Column) -> Column:
    return F.coalesce(F.element_at(mapa_tipo, token), token)


tokens = F.split(normalizar_texto(F.col("consulta_logradouro")), " ")
t1, t2 = tipo_de(F.element_at(tokens, 1)), tipo_de(F.element_at(tokens, 2))

busca = (
    busca_bronze
    .withColumn("_prefixo_lixo", ~em(TIPOS_VALIDOS, t1) & em(TIPOS_VALIDOS, t2) & (F.size(tokens) > 1))
    .withColumn("_tokens", F.when(F.col("_prefixo_lixo"), F.slice(tokens, 2, 100)).otherwise(tokens))
    .withColumn("_t1", tipo_de(F.element_at("_tokens", 1)))
    .withColumn("tipo_norm", F.when(em(TIPOS_VALIDOS, F.col("_t1")), F.col("_t1")))
    .withColumn("_resto", F.when(F.col("tipo_norm").isNotNull(), F.slice("_tokens", 2, 100)).otherwise(F.col("_tokens")))
    .withColumn("nome_norm", normalizar_nome(F.array_join("_resto", " ")))
    .withColumn("_s1", tipo_de(F.element_at("_resto", 1)))
    .withColumn("tipo_secundario",
                F.when(em(TIPOS_SECUNDARIOS, F.col("_s1")) & (F.size("_resto") > 1), F.col("_s1")))
    .withColumn("nome_alternativo",
                F.when(F.col("tipo_secundario").isNotNull(), normalizar_nome(F.array_join(F.slice("_resto", 2, 100), " "))))
    .withColumn("logradouro_norm", F.concat_ws(" ", "tipo_norm", "nome_norm"))
    .withColumn("nome_chave", chave_nome(F.col("nome_norm")))
    .withColumn("nome_fonetico", chave_fonetica(F.col("nome_chave")))
    .withColumn("nome_alternativo_chave", chave_nome(F.col("nome_alternativo")))
    # número
    .withColumn("_num_txt", F.upper(F.trim("consulta_numero")))
    .withColumn("numero", F.nullif(F.regexp_extract("_num_txt", r"^(\d+)", 1), F.lit("")).cast("int"))
    .withColumn("numero", F.when(F.col("numero") > 0, F.col("numero")))
    .withColumn("numero_sufixo", F.nullif(F.regexp_extract("_num_txt", r"^\d+\s*([A-Z]+)$", 1), F.lit("")))
    .withColumn("sem_numero", F.col("numero").isNull())
    # CEP, município
    .withColumn("cep", F.lpad(F.regexp_replace("consulta_cep", r"\D", ""), 8, "0"))
    .withColumn("cep_generico", F.col("cep").endswith("000"))
    .withColumn("cod_municipio_6", F.col("consulta_municipio"))
    # bairro e complemento
    .withColumn("_bairro", normalizar_nome(normalizar_texto(F.col("consulta_bairro"))))
    .withColumn("zona_rural", F.coalesce(em(MARCADORES_ZONA_RURAL, F.col("_bairro")), F.lit(False)))
    .withColumn("bairro_norm", F.when(~F.col("zona_rural"), F.col("_bairro")))
    .withColumn("_compl", normalizar_texto(F.col("consulta_complemento")))
    .withColumn("complemento_norm", F.when(~em(COMPLEMENTOS_SEM_VALOR, F.col("_compl")), F.col("_compl")))
    # problemas detectados (auditoria)
    .withColumn("problemas", F.array_compact(F.array(
        F.when(F.col("_prefixo_lixo") | F.upper("consulta_logradouro").startswith("NONE "), F.lit("prefixo_lixo")),
        F.when(F.col("tipo_norm").isNull(), F.lit("tipo_ausente")),
        F.when(F.col("tipo_secundario").isNotNull(), F.lit("tipo_duplicado")),
        F.when(F.col("sem_numero"), F.lit("sem_numero")),
        F.when(F.col("cep_generico"), F.lit("cep_generico")),
        F.when(F.col("bairro_norm").isNull(), F.lit("bairro_ausente")),
        F.when(F.col("zona_rural"), F.lit("zona_rural")),
    )))
    .drop("_prefixo_lixo", "_tokens", "_t1", "_resto", "_s1",
          "_num_txt", "_bairro", "_compl")
    .orderBy("source_row_number")
)

bp = busca.toPandas()
assert len(bp) == busca_bronze.count() == 107, "a limpeza não pode perder nem criar registros"
bp[["query_id", "consulta_logradouro", "tipo_norm", "nome_norm", "tipo_secundario", "nome_alternativo",
    "consulta_numero", "numero", "bairro_norm", "complemento_norm", "problemas"]]

,query_id,consulta_logradouro,tipo_norm,nome_norm,tipo_secundario,nome_alternativo,consulta_numero,numero,bairro_norm,complemento_norm,problemas
0,Q0001,RODOVIA BR 101,RODOVIA,BR 101,None,None,None,NaN,BR101,None,[sem_numero]
1,Q0002,RUA RIO BAIANO,RUA,RIO BAIANO,None,None,SN,NaN,CENTRO,None,"[sem_numero, cep_generico]"
2,Q0003,FAZENDA BICO,FAZENDA,BICO,None,None,None,NaN,None,None,"[sem_numero, cep_generico, bairro_ausente]"
3,Q0004,DISTRITO 2 DE JULHO,DISTRITO,2 DE JULHO,None,None,22,22.0,POCO CENTRAL,None,[cep_generico]
4,Q0005,RUA SERGIO BOMFIM,RUA,SERGIO BOMFIM,None,None,332,332.0,BARREIRINHAS,None,[]
5,Q0006,None EVILASIO FIRMATO,None,EVILASIO FIRMATO,None,None,28,28.0,SOCRATES REZENDE,None,"[prefixo_lixo, tipo_ausente, cep_generico]"
6,Q0007,FAZENDA LAGOA GRANDE,FAZENDA,LAGOA GRANDE,None,None,None,NaN,None,None,"[sem_numero, cep_generico, bairro_ausente]"
7,Q0008,CONJUNTO URBES NOVA,CONJUNTO,URBES NOVA,None,None,2,2.0,NOVA PASTORA,None,[cep_generico]
8,Q0009,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,CONDOMINIO,RESI DOUTOR ANTONIO CARLOS GOES,None,None,1,1.0,None,None,"[cep_generico, bairro_ausente]"
9,Q0010,RUA CASSIMIRO DE ABREU,RUA,CASSIMIRO DE ABREU,None,None,89,89.0,None,None,"[cep_generico, bairro_ausente]"


In [7]:
bp["problemas"].explode().value_counts()

problemas
cep_generico      89
bairro_ausente    44
sem_numero        36
zona_rural         9
tipo_duplicado     4
prefixo_lixo       3
tipo_ausente       1
Name: count, dtype: int64

## 4. CNEFE → silver

- Seleciona só as colunas úteis ao linkage e à auditoria.
- Confirma que cada `COD_UNICO_ENDERECO` pertence a um único setor antes de deduplicar.
- Aplica **as mesmas funções** da busca.

In [8]:
setores_por_endereco = (
    cnefe_bronze.groupBy("COD_UNICO_ENDERECO").agg(F.countDistinct("COD_SETOR").alias("n"))
    .where("n > 1").count()
)
print("endereços com mais de um setor:", setores_por_endereco)
assert setores_por_endereco == 0

endereços com mais de um setor: 0


In [9]:
cnefe = (
    cnefe_bronze
    .dropDuplicates(["COD_UNICO_ENDERECO"])
    .select(
        "COD_UNICO_ENDERECO", "cod_municipio_6", "COD_MUNICIPIO", "COD_SETOR",
        F.substring("COD_SETOR", 1, 15).alias("setor"),
        "NUM_QUADRA", "NUM_FACE",
        F.col("CEP").alias("cep"),
        F.col("CEP").endswith("000").alias("cep_generico"),
        normalizar_nome(normalizar_texto(F.col("DSC_LOCALIDADE"))).alias("localidade_norm"),
        normalizar_texto(F.col("NOM_TIPO_SEGLOGR")).alias("tipo_original_norm"),
        F.regexp_extract(normalizar_texto(F.col("NOM_TIPO_SEGLOGR")), TIPO_BASE_REGEX, 1).alias("tipo_norm"),
        normalizar_nome(normalizar_texto(F.concat_ws(" ", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR"))).alias("nome_norm"),
        (F.col("NOM_SEGLOGR") == "SEM DENOMINACAO").alias("sem_denominacao"),
        F.when(F.col("NUM_ENDERECO").cast("int") > 0, F.col("NUM_ENDERECO").cast("int")).alias("numero"),
        F.col("DSC_MODIFICADOR").alias("modificador"),
        normalizar_texto(F.col("DSC_ESTABELECIMENTO")).alias("estabelecimento_norm"),
        F.col("LATITUDE").cast("double").alias("latitude"),
        F.col("LONGITUDE").cast("double").alias("longitude"),
    )
    .withColumn("sem_numero", F.col("numero").isNull())
    .withColumn("logradouro_norm", F.concat_ws(" ", "tipo_norm", "nome_norm"))
    .withColumn("nome_chave", chave_nome(F.col("nome_norm")))
    .withColumn("nome_fonetico", chave_fonetica(F.col("nome_chave")))
)
cnefe.limit(5).toPandas()

,COD_UNICO_ENDERECO,cod_municipio_6,COD_MUNICIPIO,COD_SETOR,setor,NUM_QUADRA,NUM_FACE,cep,cep_generico,localidade_norm,...,sem_denominacao,numero,modificador,estabelecimento_norm,latitude,longitude,sem_numero,logradouro_norm,nome_chave,nome_fonetico
0,100009250,292610,2926103,292610305000008P,292610305000008,0,1,48750000,True,ALGODOES,...,False,38.0,None,None,-11.490431,-39.412332,False,POVOADO ALGODOES,ALGODOES,ALGODOES
1,100009264,292610,2926103,292610305000007P,292610305000007,0,6,48750000,True,RIACHO FUNDO,...,False,NaN,SN,None,-11.480466,-39.417251,True,FAZENDA RIACHO FUNDO,RIACHO FUNDO,RIAXO FUNDO
2,100009269,290320,2903201,290320105000296P,290320105000296,0,2,47800028,False,SITIO DA BARRIGUDA,...,False,NaN,SN,None,-12.246975,-44.836526,True,SITIO BARRIGUDA,BARRIGUDA,BARIGUDA
3,100009280,290320,2903201,290320105000297P,290320105000297,0,5,47800028,False,ARRAIAL DA PENHA,...,False,NaN,SN,None,-12.209163,-44.941235,True,POVOADO ARRAIAL DA PENHA,ARRAIAL PENHA,ARAIAL PENA
4,100009291,290320,2903201,290320105000296P,290320105000296,0,14,47800028,False,RODOVIA BR 242,...,False,NaN,SN,None,-12.171012,-44.663088,True,RODOVIA BR 242,BR 242,BR 242


## 5. Gravação da camada silver

In [10]:
(busca.coalesce(1).write.mode("overwrite").parquet(str(SILVER / "busca")))
(cnefe.repartition("cod_municipio_6").write.mode("overwrite")
      .partitionBy("cod_municipio_6").parquet(str(SILVER / "cnefe")))

busca_silver = spark.read.parquet(str(SILVER / "busca"))
cnefe_silver = spark.read.parquet(str(SILVER / "cnefe"))
print(f"busca silver: {busca_silver.count()} linhas | cnefe silver: {cnefe_silver.count():,} linhas")

busca silver: 107 linhas | cnefe silver: 8,528,506 linhas


## 6. Efeito da limpeza

Quantos registros da busca encontram **pelo menos um logradouro** com chave idêntica no mesmo município —
antes (texto bruto, EDA: 42/107) e depois da padronização. Não é o linkage: é só uma medida de quanto a limpeza
aproximou as duas bases.

In [11]:
municipios = [r[0] for r in busca_silver.select("cod_municipio_6").distinct().collect()]
logr = (cnefe_silver.where(F.col("cod_municipio_6").isin(municipios))
        .select("cod_municipio_6", "logradouro_norm", "nome_chave", "nome_fonetico").distinct().cache())


def casados(col_busca: str, col_cnefe: str) -> set[str]:
    chaves = logr.select("cod_municipio_6", F.col(col_cnefe).alias("_k")).distinct()
    j = busca_silver.join(chaves, (busca_silver.cod_municipio_6 == chaves.cod_municipio_6)
                          & (busca_silver[col_busca] == chaves["_k"]), "inner")
    return {r[0] for r in j.select("query_id").distinct().collect()}


resultado = {
    "logradouro_norm (tipo + nome)": casados("logradouro_norm", "logradouro_norm"),
    "nome_chave (sem tipo/stopwords)": casados("nome_chave", "nome_chave"),
    "nome_alternativo_chave": casados("nome_alternativo_chave", "nome_chave"),
    "nome_fonetico": casados("nome_fonetico", "nome_fonetico"),
}
uniao = set().union(*resultado.values())
resumo = pd.DataFrame({"registros casados": {k: len(v) for k, v in resultado.items()}})
resumo.loc["qualquer chave"] = len(uniao)
resumo.loc["baseline bruto (EDA)"] = 42
resumo

,registros casados
logradouro_norm (tipo + nome),46
nome_chave (sem tipo/stopwords),64
nome_alternativo_chave,0
nome_fonetico,66
qualquer chave,66
baseline bruto (EDA),42


In [12]:
# Sem nenhuma chave exata: candidatos ao passe fuzzy / rural
bp.loc[~bp["query_id"].isin(uniao), ["query_id", "consulta_logradouro", "nome_chave", "bairro_norm",
                                     "consulta_municipio", "problemas"]]

,query_id,consulta_logradouro,nome_chave,bairro_norm,consulta_municipio,problemas
2,Q0003,FAZENDA BICO,BICO,None,290200,"[sem_numero, cep_generico, bairro_ausente]"
7,Q0008,CONJUNTO URBES NOVA,URBES NOVA,NOVA PASTORA,290700,[cep_generico]
8,Q0009,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,RESI DOUTOR ANTONIO CARLOS GOES,None,290750,"[cep_generico, bairro_ausente]"
9,Q0010,RUA CASSIMIRO DE ABREU,CASSIMIRO ABREU,None,291072,"[cep_generico, bairro_ausente]"
11,Q0012,RUA PADRE PEDRO PEREIRA PROJETO II,PADRE PEDRO PEREIRA PROJETO II,None,291160,"[cep_generico, bairro_ausente]"
16,Q0017,FAZENDA PALMEIRAS DO RIACHO,PALMEIRAS RIACHO,None,291430,"[sem_numero, cep_generico, bairro_ausente, zona_rural]"
18,Q0019,RUA CAMINHO 20 CASA,CAMINHO 20 CASA,URBIS III,291560,"[tipo_duplicado, cep_generico]"
19,Q0020,FAZENDA PERSEVERANCA,PERSEVERANCA,CURVELO DA CONCEICAO,291600,"[sem_numero, cep_generico]"
23,Q0024,RUA ENTROCAMENTO DE LAJE,ENTROCAMENTO LAJE,None,291880,"[sem_numero, cep_generico, bairro_ausente, zona_rural]"
25,Q0026,FAZENDA CATANA NOVO,CATANA NOVO,None,291910,"[sem_numero, cep_generico, bairro_ausente, zona_rural]"


## 7. Conclusões

Preencher após a execução:
- [ ] Ganho da limpeza sobre o baseline de 42/107.
- [ ] Registros que dependem do passe fuzzy / rural (lista acima).
- [ ] Ajustes de dicionário sugeridos pelos não casados.

In [13]:
spark.stop()